# Analiza frekwencji
Wersja 2 — z filtrem klas piątych (do końca kwietnia), wykresami Plotly i porównaniem roczników.

In [2]:
import pandas as pd
import hashlib
import re
import plotly.express as px

## Wczytanie danych

In [3]:
df_2020_2021 = pd.read_excel("dane/frekw2020_2021.xlsx", sheet_name="Frekwencja - dane źródłowe")
df_2021_2022 = pd.read_excel("dane/frekw2021_2022.xlsx", sheet_name="Frekwencja - dane źródłowe")
df_2022_2023 = pd.read_excel("dane/frekw2022_2023.xlsx", sheet_name="Frekwencja - dane źródłowe")
df_2023_2024 = pd.read_excel("dane/frekw2023_2024.xlsx", sheet_name="Frekwencja - dane źródłowe")
df_2024_2025 = pd.read_excel("dane/frekw2024_2025.xlsx", sheet_name="Frekwencja - dane źródłowe")
df_2025_2026 = pd.read_excel("dane/frekw2025_2026.xlsx", sheet_name="Frekwencja - dane źródłowe")

## Filtrowanie — od września danego roku

In [4]:
df_2020_2021 = df_2020_2021[df_2020_2021['Data lekcji'] > '2020-09-01'].copy()
df_2021_2022 = df_2021_2022[df_2021_2022['Data lekcji'] > '2021-09-01'].copy()
df_2022_2023 = df_2022_2023[df_2022_2023['Data lekcji'] > '2022-09-01'].copy()
df_2023_2024 = df_2023_2024[df_2023_2024['Data lekcji'] > '2023-09-01'].copy()
df_2024_2025 = df_2024_2025[df_2024_2025['Data lekcji'] > '2024-09-01'].copy()
df_2025_2026 = df_2025_2026[df_2025_2026['Data lekcji'] > '2025-09-01'].copy()

## Anonimizacja uczniów

In [5]:
SALT = "moja_szkola_2026"  # NIE zmieniaj między latami!

def normalizuj(tekst):
    return " ".join(str(tekst).strip().split()).upper()

def parsuj_ucznia(uczen_str):
    return uczen_str.split(",")[0].strip()

def uczen_hash(nazwisko_imie, szkola):
    klucz = f"{SALT}|{normalizuj(szkola)}|{normalizuj(nazwisko_imie)}"
    return hashlib.sha256(klucz.encode()).hexdigest()[:10].upper()

def nauczyciel_hash(nauczyciel_str):
    klucz = f"{SALT}|NAUCZYCIEL|{normalizuj(nauczyciel_str)}"
    return "N_" + hashlib.sha256(klucz.encode()).hexdigest()[:8].upper()

def anonimizuj(df, rok_szkolny):
    df = df.copy()
    df["_nazwisko_imie"] = df["Uczeń"].apply(parsuj_ucznia)
    df["uczen_id"]       = df.apply(lambda r: uczen_hash(r["_nazwisko_imie"], r["Szkoła"]), axis=1)
    df["nauczyciel_id"]  = df["Nauczyciel"].apply(nauczyciel_hash)
    df["rok_szkolny"]    = rok_szkolny

    mapping = (
        df[["_nazwisko_imie", "Szkoła", "uczen_id"]]
        .drop_duplicates()
        .rename(columns={"_nazwisko_imie": "nazwisko_imie"})
        .sort_values("uczen_id")
        .reset_index(drop=True)
    )
    df = df.drop(columns=["Uczeń", "Nauczyciel", "_nazwisko_imie"]).copy()
    df = df[[
        "uczen_id", "rok_szkolny", "Szkoła", "Dziennik",
        "Data lekcji", "Numer pory lekcji", "Przedmiot", "Grupa", "nauczyciel_id",
        "Symbol wpisu frekwencji", "Nazwa wpisu frekwencji",
        "Kategoria typów wpisów frekwencji",
    ]]
    return df, mapping

In [6]:
df_2020_2021, mapping = anonimizuj(df_2020_2021, rok_szkolny="2020/2021")
df_2021_2022, mapping = anonimizuj(df_2021_2022, rok_szkolny="2021/2022")
df_2022_2023, mapping = anonimizuj(df_2022_2023, rok_szkolny="2022/2023")
df_2023_2024, mapping = anonimizuj(df_2023_2024, rok_szkolny="2023/2024")
df_2024_2025, mapping = anonimizuj(df_2024_2025, rok_szkolny="2024/2025")
df_2025_2026, mapping = anonimizuj(df_2025_2026, rok_szkolny="2025/2026")

## Zamiana wpisów frekwencji na kolumny dummy

In [7]:
df_list = [df_2020_2021, df_2021_2022, df_2022_2023, df_2023_2024, df_2024_2025, df_2025_2026]
df_list_ = []

for df in df_list:
    df = df.drop(columns=["Symbol wpisu frekwencji", "Kategoria typów wpisów frekwencji"]).copy()
    dummies = pd.get_dummies(df["Nazwa wpisu frekwencji"], prefix="wpis", dtype=int)
    df = pd.concat([df.drop(columns=["Nazwa wpisu frekwencji"]), dummies], axis=1)
    df_list_.append(df)

df_2020_2021, df_2021_2022, df_2022_2023, df_2023_2024, df_2024_2025, df_2025_2026 = df_list_

## Obliczanie frekwencji per uczeń

**Uwaga:** klasy piąte (numer klasy = 5) mają frekwencję liczoną tylko do końca kwietnia danego roku szkolnego.

In [8]:
kolumny_nieobecnosci = [
    'wpis_nieob. uspr. p.s.', 'wpis_nieob. uspraw.', 'wpis_nieobecność'
]
kolumny_obecnosci = [
    'wpis_nauka zdalna', 'wpis_obecność', 'wpis_spóźn. uspr.',
    'wpis_spóźnienie', 'wpis_zwolniony'
]

def czy_klasa_piata(dziennik):
    """Zwraca True jeśli numer klasy == 5 (np. 5TM, 5TŻ)."""
    numer = re.match(r"^(\d+)", str(dziennik))
    return int(numer.group(1)) == 5 if numer else False

frekwencja_dict = {}

for df in df_list_:
    rok = df["rok_szkolny"].iloc[0]
    rok_koniec = rok.split("/")[1]  # np. "2021" z "2020/2021"

    # Filtr dat: dla klas piątych tylko do końca kwietnia
    maska_piata = df["Dziennik"].apply(czy_klasa_piata)
    maska_data  = pd.to_datetime(df["Data lekcji"]) <= pd.Timestamp(f"{rok_koniec}-04-30")
    df_filtered = df[~maska_piata | maska_data].copy()

    frekwencja = (
        df_filtered
        .groupby(["rok_szkolny", "Dziennik", "uczen_id"])[kolumny_nieobecnosci + kolumny_obecnosci]
        .sum()
        .assign(
            wszystkich     = lambda x: x[kolumny_nieobecnosci + kolumny_obecnosci].sum(axis=1),
            nieobecnych    = lambda x: x[kolumny_nieobecnosci].sum(axis=1),
            frekwencja_pct = lambda x: ((x["wszystkich"] - x["nieobecnych"]) / x["wszystkich"] * 100).round(1)
        )
        .reset_index()
    )

    frekwencja_dict[rok] = frekwencja

frekwencja_2020_2021 = frekwencja_dict["2020/2021"]
frekwencja_2021_2022 = frekwencja_dict["2021/2022"]
frekwencja_2022_2023 = frekwencja_dict["2022/2023"]
frekwencja_2023_2024 = frekwencja_dict["2023/2024"]
frekwencja_2024_2025 = frekwencja_dict["2024/2025"]
frekwencja_2025_2026 = frekwencja_dict["2025/2026"]
frekwencja_list = [
    frekwencja_2020_2021, frekwencja_2021_2022, frekwencja_2022_2023,
    frekwencja_2023_2024, frekwencja_2024_2025, frekwencja_2025_2026
]

## Uczniowie z odstającą frekwencją (IQR)

In [9]:
for frekwencja in frekwencja_list:
    rok = frekwencja["rok_szkolny"].iloc[0]

    Q1 = frekwencja.groupby("Dziennik")["frekwencja_pct"].transform("quantile", 0.25)
    Q3 = frekwencja.groupby("Dziennik")["frekwencja_pct"].transform("quantile", 0.75)
    dolna = Q1 - 1.5 * (Q3 - Q1)

    odstajacy = (
        frekwencja[frekwencja["frekwencja_pct"] < dolna]
        [["Dziennik", "uczen_id", "frekwencja_pct"]]
        .sort_values(["Dziennik", "frekwencja_pct"])
        .reset_index(drop=True)
    )

    print(f"\n=== {rok} — uczniowie z odstającą frekwencją ===")
    display(odstajacy)


=== 2020/2021 — uczniowie z odstającą frekwencją ===


,Dziennik,uczen_id,frekwencja_pct
0,1TW5,E45666FE1F,71.6
1,1TŻ5,F4DC8F8F90,47.2
2,2TMR,D0B043A863,51.0
3,2TR,D0A43425CF,36.0
4,2TŻ,CDE1F65A8C,17.8



=== 2021/2022 — uczniowie z odstającą frekwencją ===


,Dziennik,uczen_id,frekwencja_pct
0,1M,1353AF652A,60.6
1,1RŻ5,77BCAF0666,33.6
2,1TM5,ADDA34429D,61.3
3,1TW5,560FDC2DD7,57.7
4,3MK,05A142B6F6,6.9
5,3MK,0A24F61788,21.0
6,3MK,89EB038FD3,36.6
7,3RŻ5,FF65D504E1,35.8
8,3TM5,2F536A29DF,51.9
9,3TŻ,8F81E96C5D,27.8



=== 2022/2023 — uczniowie z odstającą frekwencją ===


...

,Dziennik,uczen_id,frekwencja_pct
0,1M,65A4222506,24.8
1,1M,E50B981424,25.0
2,1TŻ,979FAF8236,56.3
3,1TŻ,4B94DA2FCC,59.5
4,2MK,932CFAC66E,11.5
5,2TW5,912AAEFE58,51.3
6,2TŻ5,2BDBFBD70A,50.4
7,3RŻ5,D1F6555E9E,33.2
8,3TW5,49DA0EE07B,47.7
9,4TM5,FE417FF3C3,40.1



=== 2024/2025 — uczniowie z odstającą frekwencją ===


,Dziennik,uczen_id,frekwencja_pct
0,1RŻ,1A8A6B0FC8,37.5
1,1RŻ,E677A2C610,40.6
2,1TMR,3EB3F892FB,25.4
3,1TMR,7672689A0F,29.3
4,1TW,C691BA9630,15.6
5,1TW,7B41F62E15,37.5
6,2M,D110BB1996,23.6
7,2TMR,7811B2DBCC,48.1
8,2TW,8B99A1F16C,44.5
9,2TŻ,2BDBFBD70A,40.3



=== 2025/2026 — uczniowie z odstającą frekwencją ===


,Dziennik,uczen_id,frekwencja_pct
0,1TMR,F0AB965B5B,63.3
1,1TR,F583E7B258,48.3
2,2M,A86E206FEE,36.1
3,2TMŻ,2BDBFBD70A,37.4
4,2TW,5158AD93BE,44.6
5,3M,E9C891E3F0,1.6
6,3M,6A32292DA6,1.7
7,3TW,912AAEFE58,15.7
8,3TW,8B99A1F16C,18.8
9,5TW5,EE75A467A8,5.6


## Jaka by była frekwencja klasy bez odstających uczniów


In [10]:
# Uczniowie z odstającą frekwencją
for frekwencja in frekwencja_list:
    rok = frekwencja["rok_szkolny"].iloc[0]
    frekwencja_klasy = (
        frekwencja
        .groupby("Dziennik")["frekwencja_pct"]
        .mean()
        .round(1)
        .reset_index()
        .rename(columns={"frekwencja_pct": "frekwencja_%"})
    )

    Q1 = frekwencja.groupby("Dziennik")["frekwencja_pct"].transform("quantile", 0.25)
    Q3 = frekwencja.groupby("Dziennik")["frekwencja_pct"].transform("quantile", 0.75)
    dolna = Q1 - 1.5 * (Q3 - Q1)

    odstajacy = (
        frekwencja[frekwencja["frekwencja_pct"] < dolna]
        [["Dziennik", "uczen_id", "frekwencja_pct"]]
        .sort_values(["Dziennik", "frekwencja_pct"])
        .reset_index(drop=True)
    )
    odstajacy_ids = odstajacy[["Dziennik", "uczen_id"]]

    # Frekwencja bez odstających
    frekwencja_bez = (
        frekwencja
        .merge(odstajacy_ids, on=["Dziennik", "uczen_id"], how="left", indicator=True)
        .query('_merge == "left_only"')
        .drop(columns="_merge")
    )

    # Średnia klasy bez odstających
    frekwencja_klasy_bez = (
        frekwencja_bez
        .groupby("Dziennik")["frekwencja_pct"]
        .mean()
        .round(1)
        .reset_index()
        .rename(columns={"frekwencja_pct": "frekwencja_bez_odstajacych_%"})
    )

    # Porównanie
    porownanie = (
        frekwencja_klasy
        .merge(frekwencja_klasy_bez, on="Dziennik")
        .assign(roznica=lambda x: (x["frekwencja_bez_odstajacych_%"] - x["frekwencja_%"]).round(1))
        .sort_values("roznica", ascending=False)
    )
    print(f"\n=== {rok} — porównanie średniej frekwencji klasy z i bez odstających ===")
    display(porownanie)


=== 2020/2021 — porównanie średniej frekwencji klasy z i bez odstających ===


,Dziennik,frekwencja_%,frekwencja_bez_odstajacych_%,roznica
9,2TŻ,81.6,84.5,2.9
3,1TŻ5,83.2,85.7,2.5
8,2TR,75.4,77.8,2.4
2,1TW5,91.1,92.6,1.5
7,2TMR,78.6,79.9,1.3
0,1M,74.0,74.0,0.0
1,1TM5,85.2,85.2,0.0
4,2MK,70.8,70.8,0.0
5,2RŻ5,77.2,77.2,0.0
6,2TM5,85.0,85.0,0.0



=== 2021/2022 — porównanie średniej frekwencji klasy z i bez odstających ===


,Dziennik,frekwencja_%,frekwencja_bez_odstajacych_%,roznica
8,3MK,62.1,68.2,6.1
15,4TŻ,73.7,76.4,2.7
13,3TŻ,69.8,71.9,2.1
1,1RŻ5,81.0,82.7,1.7
0,1M,78.5,79.7,1.2
10,3TM5,76.4,77.6,1.2
9,3RŻ5,69.7,70.8,1.1
2,1TM5,83.1,84.1,1.0
3,1TW5,81.1,82.1,1.0
4,2M,61.9,61.9,0.0



=== 2022/2023 — porównanie średniej frekwencji klasy z i bez odstających ===


...

,Dziennik,frekwencja_%,frekwencja_bez_odstajacych_%,roznica
15,4TW5,77.6,83.5,5.9
0,1M,68.0,71.7,3.7
5,2MK,64.6,66.3,1.7
4,1TŻ,81.0,82.7,1.7
11,3RŻ5,71.6,73.1,1.5
18,5TM5,73.8,75.2,1.4
13,3TW5,75.1,76.4,1.3
14,4TM5,62.5,63.8,1.3
9,2TŻ5,75.3,76.5,1.2
8,2TW5,76.0,77.1,1.1



=== 2024/2025 — porównanie średniej frekwencji klasy z i bez odstających ===


,Dziennik,frekwencja_%,frekwencja_bez_odstajacych_%,roznica
18,5TW5,68.0,76.2,8.2
3,1TW,73.2,78.4,5.2
2,1TMR,73.5,78.1,4.6
1,1RŻ,74.8,79.0,4.2
19,5TŻ5,69.7,72.5,2.8
11,3TR5,67.7,70.0,2.3
8,2TŻ,76.5,78.7,2.2
4,2M,66.2,68.2,2.0
16,4TW5,75.9,77.3,1.4
17,5TM5,57.7,59.0,1.3



=== 2025/2026 — porównanie średniej frekwencji klasy z i bez odstających ===


,Dziennik,frekwencja_%,frekwencja_bez_odstajacych_%,roznica
8,3M,54.7,59.3,4.6
11,3TW,60.8,64.6,3.8
19,5TW5,65.5,68.2,2.7
5,2M,68.8,70.8,2.0
7,2TW,73.9,75.6,1.7
2,1TR,77.1,78.5,1.4
6,2TMŻ,72.6,73.9,1.3
1,1TMR,80.6,81.4,0.8
14,4TR5,65.6,65.6,0.0
18,5TM5,58.8,58.8,0.0


## Porównanie frekwencji klas przez lata

### Budowanie df_porownanie

In [11]:
def dziennik_na_rocznik(dziennik, rok_szkolny):
    """Oblicza rok rozpoczęcia nauki na podstawie numeru klasy i roku szkolnego."""
    rok_start = int(rok_szkolny.split("/")[0])
    numer = re.match(r"^(\d+)", str(dziennik))
    if numer:
        nr = int(numer.group(1))
        return rok_start - nr + 1
    return None

wszystkie_lata = []

for frekwencja in frekwencja_list:
    rok = frekwencja["rok_szkolny"].iloc[0]
    frekwencja_klasy = (
        frekwencja
        .groupby("Dziennik")["frekwencja_pct"]
        .mean()
        .round(1)
        .reset_index()
        .rename(columns={"frekwencja_pct": "frekwencja_%"})
    )
    frekwencja_klasy["rok_szkolny"] = rok
    wszystkie_lata.append(frekwencja_klasy)

df_porownanie = pd.concat(wszystkie_lata, ignore_index=True)
df_porownanie["rocznik"] = df_porownanie.apply(
    lambda r: dziennik_na_rocznik(r["Dziennik"], r["rok_szkolny"]), axis=1
)
df_porownanie["sufiks"] = df_porownanie["Dziennik"].str.extract(r"^\d+(.*)")
df_porownanie["rocznik_str"] = df_porownanie["rocznik"].astype(str)

df_porownanie.head()

,Dziennik,frekwencja_%,rok_szkolny,rocznik,sufiks,rocznik_str
0,1M,74.0,2020/2021,2020,M,2020
1,1TM5,85.2,2020/2021,2020,TM5,2020
2,1TW5,91.1,2020/2021,2020,TW5,2020
3,1TŻ5,83.2,2020/2021,2020,TŻ5,2020
4,2MK,70.8,2020/2021,2019,MK,2019


## Wykresy: każdy rocznik — klasy przez lata

In [12]:
for rocznik, grupa in df_porownanie.groupby("rocznik"):
    grupa = grupa.sort_values(["rok_szkolny", "sufiks"])

    fig = px.bar(
        grupa,
        x="rok_szkolny",
        y="frekwencja_%",
        color="sufiks",
        barmode="group",
        text="Dziennik",
        hover_data={"Dziennik": True, "frekwencja_%": True, "rocznik_str": True},
        title=f"Rocznik {rocznik} — frekwencja klas przez lata"
    )
    fig.update_traces(textangle=45, textposition="outside")
    fig.update_layout(
        yaxis_range=[50, 105],
        xaxis_title="Rok szkolny",
        yaxis_title="Frekwencja [%]",
        legend_title="Kierunek",
        height=500
    )
    fig.show()

## Wykresy: każdy kierunek — przez wszystkie lata i roczniki

In [13]:
for sufiks, grupa in df_porownanie.groupby("sufiks"):
    grupa = grupa.sort_values(["rok_szkolny", "rocznik"])

    fig = px.bar(
        grupa,
        x="rok_szkolny",
        y="frekwencja_%",
        color="rocznik_str",
        barmode="group",
        text="Dziennik",
        hover_data={"Dziennik": True, "frekwencja_%": True, "rocznik_str": True},
        title=f"Kierunek {sufiks} — frekwencja przez wszystkie lata"
    )
    fig.update_traces(textangle=45, textposition="outside")
    fig.update_layout(
        yaxis_range=[50, 105],
        xaxis_title="Rok szkolny",
        yaxis_title="Frekwencja [%]",
        legend_title="Rocznik",
        height=500
    )
    fig.show()
    

In [14]:
# Połącz surowe dane wszystkich lat z rocznikiem
df_all = pd.concat(df_list_, ignore_index=True)

# Dodaj numer dnia tygodnia
df_all["dzien_tygodnia"] = pd.to_datetime(df_all["Data lekcji"]).dt.dayofweek  # 0=pon, 4=pt

# Dodaj sufiks i rocznik
df_all["sufiks"] = df_all["Dziennik"].str.extract(r"^\d+(.*)")
df_all["rocznik"] = df_all.apply(
    lambda r: dziennik_na_rocznik(r["Dziennik"], r["rok_szkolny"]), axis=1
)

DNI = {0: "Poniedziałek", 1: "Wtorek", 2: "Środa", 3: "Czwartek", 4: "Piątek"}

# Frekwencja % per rocznik + dzień tygodnia (po wszystkich latach łącznie)
wpis_cols_nieob = ['wpis_nieob. uspr. p.s.', 'wpis_nieob. uspraw.', 'wpis_nieobecność']
wpis_cols_wszystkie = wpis_cols_nieob + [
    'wpis_nauka zdalna', 'wpis_obecność', 'wpis_spóźn. uspr.',
    'wpis_spóźnienie', 'wpis_zwolniony'
]

frekw_dzien_rok = (
    df_all
    .groupby(["rok_szkolny", "Dziennik", "dzien_tygodnia"])[wpis_cols_wszystkie]
    .sum()
    .assign(
        wszystkich     = lambda x: x[wpis_cols_wszystkie].sum(axis=1),
        nieobecnych    = lambda x: x[wpis_cols_nieob].sum(axis=1),
        frekwencja_pct = lambda x: ((x["wszystkich"] - x["nieobecnych"]) / x["wszystkich"] * 100).round(1)
    )
    .reset_index()
)
frekw_dzien_rok["dzien_nazwa"] = frekw_dzien_rok["dzien_tygodnia"].map(DNI)

for rok_szkolny, grupa in frekw_dzien_rok.groupby("rok_szkolny"):
    grupa = grupa.sort_values("dzien_tygodnia")

    maska_branzowa = grupa["Dziennik"].str.match(r"^\d+(M|MK|RŻ\d*)")
    branzowa = grupa[maska_branzowa]
    technikum = grupa[~maska_branzowa]

    for nazwa, podgrupa in [("Branżowa", branzowa), ("Technikum", technikum)]:
        if podgrupa.empty:
            continue
        n_klas = podgrupa["Dziennik"].nunique()

        fig = px.bar(
            podgrupa,
            x="dzien_nazwa",
            y="frekwencja_pct",
            color="Dziennik",
            barmode="group",
            text="frekwencja_pct",
            title=f"{rok_szkolny} — {nazwa} — frekwencja wg dnia tygodnia",
            labels={
                "dzien_nazwa": "Dzień tygodnia",
                "frekwencja_pct": "Frekwencja [%]",
                "Dziennik": "Klasa"
            },
            category_orders={
                "dzien_nazwa": ["Poniedziałek", "Wtorek", "Środa", "Czwartek", "Piątek"]
            }
        )
        fig.update_traces(textposition="outside", textangle=0, textfont_size=8)
        fig.update_layout(
            yaxis_range=[50, 90],
            height=500,
            width=min(300 + n_klas * 80, 1200),
            bargap=0.1,
            bargroupgap=0.05,
            legend=dict(orientation="v", x=1.02, y=1)
        )
        fig.show()

        # Tabela pod wykresem
        tabela = (
            podgrupa
            .pivot(index="Dziennik", columns="dzien_nazwa", values="frekwencja_pct")
            .reindex(columns=["Poniedziałek", "Wtorek", "Środa", "Czwartek", "Piątek"])
            .sort_index()
        )
        tabela.columns.name = None
        tabela.index.name = "Klasa"
        tabela["Średnia"] = tabela.mean(axis=1).round(1)
        display(tabela)

,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1M,72.8,72.8,77.6,72.7,74.6,74.1
2MK,60.6,58.7,75.2,69.7,69.6,66.8
2RŻ5,73.6,77.9,75.8,74.9,81.3,76.7
3M,54.0,55.8,48.1,46.5,58.3,52.5


,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1TM5,73.8,88.7,86.0,81.4,83.9,82.8
1TW5,91.1,90.2,93.4,89.5,90.4,90.9
1TŻ5,83.5,89.5,82.4,81.8,81.3,83.7
2TM5,87.4,82.5,84.0,85.5,83.4,84.6
2TMR,79.4,80.2,79.5,78.6,75.3,78.6
2TR,76.6,76.9,80.5,73.6,69.2,75.4
2TŻ,85.5,85.9,83.0,84.3,76.7,83.1
3TRM,77.9,78.4,74.9,72.4,78.0,76.3
3TŻ,84.8,83.8,80.0,84.6,87.0,84.0


,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1M,78.6,78.0,82.0,79.2,71.7,77.9
1RŻ5,80.8,85.8,81.6,78.2,77.6,80.8
2M,61.1,67.6,61.6,65.2,57.2,62.5
3MK,50.8,69.1,64.1,62.9,62.4,61.9
3RŻ5,70.7,71.6,70.8,47.2,68.3,65.7


,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1TM5,83.9,85.3,81.9,87.2,77.1,83.1
1TW5,79.0,83.2,81.2,80.5,80.9,81.0
2TM5,78.4,81.3,79.2,79.6,62.7,76.2
2TW5,78.4,79.4,80.7,76.1,78.0,78.5
2TŻ5,71.6,79.3,78.4,74.9,72.1,75.3
3TM5,75.5,80.4,75.4,75.0,74.7,76.2
3TMR,64.5,71.1,64.3,63.0,67.9,66.2
3TR,75.1,72.3,74.0,67.2,68.1,71.3
3TŻ,68.9,73.0,72.4,73.7,63.2,70.2


,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1MK,78.9,78.7,76.1,75.6,79.2,77.7
2M,66.3,72.6,69.2,70.9,64.3,68.7
2RŻ5,80.3,81.9,81.9,78.7,72.6,79.1
3M,62.5,61.6,71.4,61.0,54.9,62.3
4RŻ5,67.1,68.7,64.6,62.8,62.0,65.0


...

,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1TMR,85.0,76.8,80.1,74.5,75.5,78.4
1TR,84.4,77.9,77.0,74.8,69.8,76.8
1TW,81.2,80.1,78.8,74.6,76.9,78.3
1TŻ,83.3,83.2,85.6,75.3,76.6,80.8
2TM5,82.3,78.4,71.2,67.2,64.6,72.7
2TR5,79.3,73.7,75.3,70.2,71.7,74.0
2TW5,80.8,78.2,78.5,69.4,72.0,75.8
2TŻ5,79.1,76.5,78.0,73.7,69.7,75.4
3TM5,69.2,70.8,71.2,66.2,63.8,68.2


,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1M,73.5,72.3,69.5,68.2,66.7,70.0
1RŻ,72.8,77.2,76.6,77.0,65.6,73.8
2M,68.1,69.9,65.5,66.3,61.7,66.3
3MK,60.6,74.7,56.9,61.1,54.0,61.5
4RŻ5,72.2,70.2,66.6,69.7,58.6,67.5


,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1TMR,80.7,82.1,80.0,80.7,68.2,78.3
1TW,79.3,83.5,78.0,77.3,70.5,77.7
2TMR,73.3,73.9,76.7,69.0,71.5,72.9
2TR,73.1,78.0,71.8,68.4,67.9,71.8
2TW,71.6,71.9,73.5,69.5,68.5,71.0
2TŻ,77.4,79.6,78.5,76.3,72.0,76.8
3TM5,71.1,69.3,57.2,68.4,57.4,64.7
3TR5,72.1,69.3,71.0,66.5,57.9,67.4
3TW5,72.5,68.0,69.8,65.0,63.9,67.8


,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1M,65.6,60.0,63.6,61.1,50.6,60.2
2M,76.5,68.8,73.6,70.0,64.1,70.6
3M,58.7,60.5,60.2,59.0,55.3,58.7
5RŻ5,66.9,70.7,68.5,69.1,60.2,67.1


,Poniedziałek,Wtorek,Środa,Czwartek,Piątek,Średnia
Klasa,,,,,,
1TMR,83.3,83.4,84.4,79.9,73.2,80.8
1TR,81.0,81.0,80.2,80.1,67.5,78.0
1TW,84.8,85.6,81.0,81.5,75.1,81.6
1TŻ,76.8,79.3,77.4,70.6,68.5,74.5
2TMŻ,77.2,78.0,75.0,71.5,64.4,73.2
2TW,75.0,76.4,78.4,72.9,66.9,73.9
3TMR,64.1,67.1,60.2,60.3,58.9,62.1
3TR,70.0,70.8,68.9,65.0,54.4,65.8
3TW,68.0,68.6,67.4,57.6,49.7,62.3


In [15]:
print(df_all[df_all["rok_szkolny"] == "2025/2026"][["Dziennik", "rocznik"]].drop_duplicates().sort_values("Dziennik"))

        Dziennik  rocznik
2531217       1M     2025
2111188     1TMR     2025
2139566      1TR     2025
2163858      1TW     2025
2195823      1TŻ     2025
2549644       2M     2024
2225886     2TMŻ     2024
2259506      2TW     2024
2568284       3M     2023
2281521     3TMR     2023
2308995      3TR     2023
2334310      3TW     2023
2364883      3TŻ     2023
2395665     4TM5     2022
2417404     4TR5     2022
2433638     4TW5     2022
2457682     4TŻ5     2022
2477775     5RŻ5     2021
2496273     5TM5     2021
2513902     5TW5     2021


In [16]:
print(df_all.groupby("rocznik")["Dziennik"].nunique().sort_index())

rocznik
2017     2
2018     5
2019    19
2020    18
2021    18
2022    19
2023    15
2024     7
2025     5
Name: Dziennik, dtype: int64


## Dashboard dla dyrektora (HTML) — osobny plik na każdy rok szkolny
Eksport do samodzielnych plików HTML (jeden na rok szkolny), z podsumowaniem KPI, trendami wieloletnimi (z podświetleniem bieżącego roku), rankingiem klas i alertami uczniów. Każdy plik ma link nawigacyjny do pozostałych lat.

Wymaga: `plotly` (już używanej wyżej).


In [17]:
"""
Generator dashboardu frekwencji dla dyrektora — JEDEN plik HTML NA KAŻDY ROK SZKOLNY.
Wejście: frekwencja_list (lista df per rok), df_porownanie (frekwencja klas przez lata).
Wyjście: dashboard_frekwencja_<rok>.html dla każdego roku + dashboard_frekwencja_index.html (spis treści).
"""
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime


PROG_ALERT   = 85.0
KOLOR_DOBRA  = "#4C7A6B"
KOLOR_UWAGA  = "#C98A2C"
KOLOR_ALARM  = "#A8433A"
KOLOR_NAVY   = "#1C2B42"
KOLOR_PAPIER = "#F6F3EC"
KOLOR_LINIA  = "#DCD5C4"
KOLOR_TEKST2 = "#5B5F66"

def kolor_dla(pct):
    if pct >= 90: return KOLOR_DOBRA
    if pct >= PROG_ALERT: return KOLOR_UWAGA
    return KOLOR_ALARM

lata_posortowane = sorted(df_porownanie["rok_szkolny"].unique())

# uczniowie odstający — policzone raz, dla wszystkich lat naraz
odstajacy_wszystkie = []
for frekwencja in frekwencja_list:
    rok = frekwencja["rok_szkolny"].iloc[0]
    Q1 = frekwencja.groupby("Dziennik")["frekwencja_pct"].transform("quantile", 0.25)
    Q3 = frekwencja.groupby("Dziennik")["frekwencja_pct"].transform("quantile", 0.75)
    dolna = Q1 - 1.5 * (Q3 - Q1)
    odst = frekwencja[frekwencja["frekwencja_pct"] < dolna][["Dziennik", "uczen_id", "frekwencja_pct"]].copy()
    odst["rok_szkolny"] = rok
    odstajacy_wszystkie.append(odst)
odstajacy_all = pd.concat(odstajacy_wszystkie, ignore_index=True)

def linechart(df, grupa_col, tytul, podswietl_rok):
    fig = go.Figure()
    palette = ["#1C2B42","#4C7A6B","#C98A2C","#A8433A","#7C6A9C","#3E7CB1","#8C8266","#5B5F66"]
    grupy = sorted(df[grupa_col].dropna().unique(), key=lambda x: str(x))
    for i, g in enumerate(grupy):
        sub = df[df[grupa_col] == g].groupby("rok_szkolny")["frekwencja_%"].mean().reindex(lata_posortowane)
        fig.add_trace(go.Scatter(
            x=lata_posortowane, y=sub.values, mode="lines+markers",
            name=str(g), line=dict(width=2.5, color=palette[i % len(palette)]),
            marker=dict(size=6)
        ))
    # pionowa linia / pas podświetlający wybrany rok
    fig.add_vline(x=podswietl_rok, line_width=14, line_color="rgba(28,43,66,0.07)")
    fig.update_layout(
        title=dict(text=tytul, font=dict(family="Lora, serif", size=17, color=KOLOR_NAVY)),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Inter, sans-serif", size=12, color=KOLOR_TEKST2),
        margin=dict(t=50, l=40, r=20, b=40), height=340,
        yaxis=dict(title="Frekwencja [%]", gridcolor=KOLOR_LINIA, range=[50,100]),
        xaxis=dict(gridcolor=KOLOR_LINIA),
        legend=dict(orientation="h", y=-0.2, font=dict(size=10)),
        hovermode="x unified"
    )
    return fig

def nazwa_pliku(rok):
    return f"dashboard_frekwencja_{rok.replace('/', '_')}.html"

def zbuduj_dashboard(rok, sciezka_wyjsciowa="."):
    idx = lata_posortowane.index(rok)
    poprzedni_rok = lata_posortowane[idx - 1] if idx > 0 else None

    frekwencja_rok = next(f for f in frekwencja_list if f["rok_szkolny"].iloc[0] == rok)
    frekwencja_poprzedni = next((f for f in frekwencja_list if poprzedni_rok and f["rok_szkolny"].iloc[0] == poprzedni_rok), None)

    srednia_rok = frekwencja_rok["frekwencja_pct"].mean()
    srednia_poprzedni = frekwencja_poprzedni["frekwencja_pct"].mean() if frekwencja_poprzedni is not None else None
    delta = (srednia_rok - srednia_poprzedni) if srednia_poprzedni is not None else None

    odstajacy_rok = odstajacy_all[odstajacy_all["rok_szkolny"] == rok].sort_values(["Dziennik", "frekwencja_pct"])
    liczba_odstajacych = len(odstajacy_rok)

    ranking_rok = df_porownanie[df_porownanie["rok_szkolny"] == rok].sort_values("frekwencja_%")
    liczba_ponizej_progu = (ranking_rok["frekwencja_%"] < PROG_ALERT).sum()

    fig_rocznik = linechart(df_porownanie, "rocznik_str", "Frekwencja wg rocznika — trend przez lata", rok)
    fig_kierunek = linechart(df_porownanie, "sufiks", "Frekwencja wg kierunku — trend przez lata", rok)
    chart_rocznik_html = fig_rocznik.to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})
    chart_kierunek_html = fig_kierunek.to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})

    def wiersz_rankingu(i, row):
        kolor = kolor_dla(row["frekwencja_%"])
        szerokosc = max(min(row["frekwencja_%"], 100), 0)
        return f"""
        <tr>
          <td class="lp">{i}</td>
          <td class="klasa">{row['Dziennik']}</td>
          <td class="bar-cell"><div class="bar-track"><div class="bar-fill" style="width:{szerokosc}%; background:{kolor};"></div></div></td>
          <td class="wartosc" style="color:{kolor};">{row['frekwencja_%']:.1f}%</td>
        </tr>"""
    wiersze_ranking = "\n".join(wiersz_rankingu(i+1, row) for i, (_, row) in enumerate(ranking_rok.iterrows()))

    def alert_karta(dziennik, grupa):
        wiersze = "".join(
            f'<div class="alert-row"><span class="alert-id">{r.uczen_id}</span>'
            f'<span class="alert-pct" style="color:{KOLOR_ALARM}">{r.frekwencja_pct:.1f}%</span></div>'
            for r in grupa.itertuples()
        )
        return f'<div class="alert-card"><div class="alert-header">{dziennik} <span class="alert-count">{len(grupa)}</span></div>{wiersze}</div>'

    if len(odstajacy_rok):
        alerty_html = "".join(alert_karta(dz, g) for dz, g in odstajacy_rok.groupby("Dziennik"))
    else:
        alerty_html = '<p class="brak-alertow">Brak uczniów z istotnie odstającą frekwencją w tym roku.</p>'

    if delta is not None:
        kolor_delta = KOLOR_DOBRA if delta >= 0 else KOLOR_ALARM
        strzalka = "▲" if delta >= 0 else "▼"
        delta_html = f'<span style="color:{kolor_delta}">{strzalka} {delta:+.1f} pkt proc.</span> vs {poprzedni_rok}'
    else:
        delta_html = "brak danych porównawczych (pierwszy rok w zestawieniu)"

    inne_lata = [r for r in lata_posortowane if r != rok]
    linki_lat = " · ".join(f'<a href="{nazwa_pliku(r)}">{r}</a>' for r in inne_lata)

    kpi_html = f"""
    <div class="kpi-grid">
      <div class="kpi-card"><div class="kpi-label">Średnia frekwencja szkoły</div>
        <div class="kpi-value">{srednia_rok:.1f}%</div><div class="kpi-sub">{delta_html}</div></div>
      <div class="kpi-card"><div class="kpi-label">Rok szkolny</div>
        <div class="kpi-value" style="font-size:28px;">{rok}</div><div class="kpi-sub">okres analizy</div></div>
      <div class="kpi-card"><div class="kpi-label">Uczniowie z odstającą frekwencją</div>
        <div class="kpi-value" style="color:{KOLOR_ALARM if liczba_odstajacych else KOLOR_DOBRA};">{liczba_odstajacych}</div>
        <div class="kpi-sub">wg metody IQR, ten rok</div></div>
      <div class="kpi-card"><div class="kpi-label">Klasy poniżej {PROG_ALERT:.0f}%</div>
        <div class="kpi-value" style="color:{KOLOR_ALARM if liczba_ponizej_progu else KOLOR_DOBRA};">{liczba_ponizej_progu} / {len(ranking_rok)}</div>
        <div class="kpi-sub">wymagają uwagi wychowawcy</div></div>
    </div>"""

    wygenerowano = datetime.now().strftime("%d.%m.%Y %H:%M")

    html = f"""<!DOCTYPE html>
<html lang="pl">
<head>
<meta charset="UTF-8">
<title>Frekwencja {rok} — przegląd dla dyrektora</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Lora:wght@500;600;700&family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
<style>
  :root {{ --navy:{KOLOR_NAVY}; --paper:{KOLOR_PAPIER}; --line:{KOLOR_LINIA}; --text2:{KOLOR_TEKST2}; --dobra:{KOLOR_DOBRA}; --uwaga:{KOLOR_UWAGA}; --alarm:{KOLOR_ALARM}; }}
  * {{ box-sizing: border-box; }}
  body {{ margin:0; background:var(--paper); color:var(--navy); font-family:'Inter',sans-serif; padding:0 0 60px 0; }}
  .wrap {{ max-width:1100px; margin:0 auto; padding:0 32px; }}
  header {{ border-bottom:3px double var(--navy); padding:36px 0 20px 0; margin-bottom:32px; display:flex; justify-content:space-between; align-items:flex-end; flex-wrap:wrap; gap:12px;}}
  header .eyebrow {{ font-size:12px; letter-spacing:.12em; text-transform:uppercase; color:var(--text2); margin-bottom:6px; }}
  header h1 {{ font-family:'Lora',serif; font-weight:600; font-size:32px; margin:0 0 8px 0; }}
  header .meta {{ font-size:13px; color:var(--text2); }}
  header .lata-nav {{ font-size:13px; }}
  header .lata-nav a {{ color: var(--navy); }}
  .kpi-grid {{ display:grid; grid-template-columns:repeat(4,1fr); gap:16px; margin-bottom:44px; }}
  .kpi-card {{ background:#fff; border:1px solid var(--line); border-radius:4px; padding:18px 20px; position:relative; }}
  .kpi-card::before {{ content:""; position:absolute; top:0; left:0; width:3px; height:100%; background:var(--navy); }}
  .kpi-label {{ font-size:12px; color:var(--text2); margin-bottom:8px; text-transform:uppercase; letter-spacing:.04em;}}
  .kpi-value {{ font-family:'Lora',serif; font-size:34px; font-weight:600; line-height:1; }}
  .kpi-sub {{ font-size:12px; color:var(--text2); margin-top:8px; }}
  section {{ margin-bottom:48px; }}
  section h2 {{ font-family:'Lora',serif; font-size:20px; font-weight:600; border-bottom:1px solid var(--line); padding-bottom:10px; margin-bottom:20px; }}
  .charts-row {{ display:grid; grid-template-columns:1fr 1fr; gap:24px; }}
  .chart-box {{ background:#fff; border:1px solid var(--line); border-radius:4px; padding:8px 12px; }}
  table.ranking {{ width:100%; border-collapse:collapse; background:#fff; }}
  table.ranking th {{ text-align:left; font-size:11px; text-transform:uppercase; letter-spacing:.05em; color:var(--text2); padding:8px 12px; border-bottom:2px solid var(--navy); }}
  table.ranking td {{ padding:9px 12px; border-bottom:1px solid var(--line); font-size:14px; }}
  table.ranking td.lp {{ color:var(--text2); width:30px; font-variant-numeric:tabular-nums; }}
  table.ranking td.klasa {{ font-weight:600; width:90px; }}
  table.ranking td.wartosc {{ text-align:right; font-weight:600; width:70px; font-variant-numeric:tabular-nums; }}
  .bar-cell {{ width:100%; }}
  .bar-track {{ background:var(--line); border-radius:3px; height:10px; overflow:hidden; }}
  .bar-fill {{ height:100%; border-radius:3px; }}
  .alert-grid {{ display:grid; grid-template-columns:repeat(auto-fill,minmax(220px,1fr)); gap:14px; }}
  .alert-card {{ background:#fff; border:1px solid var(--alarm); border-left:4px solid var(--alarm); border-radius:4px; padding:14px 16px; }}
  .alert-header {{ font-weight:700; margin-bottom:8px; display:flex; justify-content:space-between; align-items:center;}}
  .alert-count {{ background:var(--alarm); color:#fff; font-size:11px; padding:1px 7px; border-radius:10px; }}
  .alert-row {{ display:flex; justify-content:space-between; font-size:13px; padding:3px 0; border-top:1px dashed var(--line); }}
  .alert-row:first-of-type {{ border-top:none; }}
  .alert-id {{ color:var(--text2); font-variant-numeric:tabular-nums; }}
  .alert-pct {{ font-weight:600; }}
  .brak-alertow {{ color:var(--dobra); font-weight:600; }}
  footer {{ text-align:center; font-size:12px; color:var(--text2); margin-top:40px; }}
  @media (max-width:800px) {{ .kpi-grid {{ grid-template-columns:repeat(2,1fr); }} .charts-row {{ grid-template-columns:1fr; }} }}
</style>
</head>
<body>
<div class="wrap">
  <header>
    <div>
      <div class="eyebrow">Dziennik elektroniczny — moduł frekwencji</div>
      <h1>Przegląd frekwencji dla dyrektora</h1>
      <div class="meta">Rok szkolny {rok} · dane anonimizowane · wygenerowano {wygenerowano}</div>
    </div>
    <div class="lata-nav">Inne lata: {linki_lat}</div>
  </header>
  {kpi_html}
  <section>
    <h2>Trendy wieloletnie</h2>
    <div class="charts-row">
      <div class="chart-box">{chart_rocznik_html}</div>
      <div class="chart-box">{chart_kierunek_html}</div>
    </div>
  </section>
  <section>
    <h2>Ranking klas — rok {rok}</h2>
    <table class="ranking">
      <thead><tr><th>Lp.</th><th>Klasa</th><th></th><th>Frekwencja</th></tr></thead>
      <tbody>{wiersze_ranking}</tbody>
    </table>
  </section>
  <section>
    <h2>Alerty — uczniowie z odstającą frekwencją ({rok})</h2>
    <div class="alert-grid">{alerty_html}</div>
  </section>
  <footer>Dane pochodzą z dziennika elektronicznego. Identyfikatory uczniów są zanonimizowane (SHA-256).</footer>
</div>
</body>
</html>"""

    sciezka = f"{sciezka_wyjsciowa}/{nazwa_pliku(rok)}"
    with open(sciezka, "w", encoding="utf-8") as f:
        f.write(html)
    return sciezka

# ---- wygeneruj dashboard dla KAŻDEGO roku szkolnego ----
for rok in lata_posortowane:
    sciezka = zbuduj_dashboard(rok, sciezka_wyjsciowa=".")
    print("Zapisano:", sciezka)


Zapisano: ./dashboard_frekwencja_2020_2021.html
Zapisano: ./dashboard_frekwencja_2021_2022.html
Zapisano: ./dashboard_frekwencja_2022_2023.html
Zapisano: ./dashboard_frekwencja_2023_2024.html
Zapisano: ./dashboard_frekwencja_2024_2025.html
Zapisano: ./dashboard_frekwencja_2025_2026.html
